# 🎵 Analyse et Traitement Audio Industriel

Ce notebook permet d'analyser des fichiers audio (sons industriels, voix) et de les mixer avec un contrôle précis du rapport signal/bruit (SNR).

**Auteur** : Romuald  
**Projet** : Mémoire des Territoires - Analyse audio industrielle  
**Date** : Janvier 2026

---

## 📚 Table des matières

1. [Configuration et imports](#config)
2. [Fonctions d'analyse audio](#analysis)
3. [Analyse d'un fichier audio](#demo-analysis)
4. [Réduction de volume](#volume)
5. [Mixage voix + bruit de fond](#mixing)

In [ ]:
# Cellule 1 : Imports et configuration
"""
═══════════════════════════════════════════════════════════════
Configuration du notebook et imports des bibliothèques
═══════════════════════════════════════════════════════════════
"""

from pathlib import Path
import librosa
import soundfile as sf
import numpy as np
import warnings

# Supprimer les warnings non critiques pour une sortie plus propre
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Configuration des chemins
DATA_DIR = Path.cwd().parent.parent / "data"
AUDIO_INDUSTRIAL_DIR = DATA_DIR / "audio" / "background_sounds" / "Pont roulant"
AUDIO_VOICE_DIR = DATA_DIR / "generated_speech"
OUTPUT_DIR = DATA_DIR / "discours"

# Créer les dossiers de sortie s'ils n'existent pas
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Vérification
print("✓ Configuration chargée")
print(f"  Répertoire data: {DATA_DIR}")
print(f"  Dossier sortie: {OUTPUT_DIR}")
print(f"  Exists: {DATA_DIR.exists()}")


✓ Configuration chargée
  Répertoire data: /Users/julienrm/Workspace/Lab_IA/memoiredesterritoires/data
  Dossier sortie: /Users/julienrm/Workspace/Lab_IA/memoiredesterritoires/data/discours
  Exists: True


## 🔬 Fonctions d'analyse audio

Fonctions d'interprétation des métriques audio pour des sons industriels.

## Informations temporelles

### Durée
La longueur totale de l'enregistrement en secondes. C'est simplement le temps total du fichier audio.

**Formule** : `Durée = Nombre d'échantillons / Taux d'échantillonnage`

### Taux d'échantillonnage (Sample Rate)
Le nombre de mesures (échantillons) prises par seconde, exprimé en Hertz (Hz).

**Standards courants** :
- 44100 Hz : qualité CD audio
- 48000 Hz : production audio/vidéo professionnelle
- 16000 Hz : téléphonie, reconnaissance vocale
- 8000 Hz : téléphonie basique

**Théorème de Shannon** : Le taux d'échantillonnage doit être au moins le double de la fréquence maximale à capturer. Avec 44100 Hz, on peut représenter des fréquences jusqu'à 22050 Hz (limite de l'audition humaine ≈ 20000 Hz).

### Nombre d'échantillons
Le total de points de données numériques constituant le signal audio.

**Calcul** : `Échantillons = Durée (s) × Taux d'échantillonnage (Hz)`

---

## Intensité et niveau sonore

### RMS (Root Mean Square)
Mesure du niveau sonore moyen d'un signal, exprimé en décibels (dB).

**Signification** : 
- 0 dB = niveau maximum possible (saturation)
- -6 dB = signal de niveau moyen
- -20 à -40 dB = signal faible
- < -60 dB = quasi-silence

**Interprétation** :
- RMS élevé → son fort/puissant
- RMS faible → son doux/faible
- Variation de RMS dans le temps → dynamique du signal

---

## Analyse fréquentielle

### F0 - Fréquence fondamentale
La fréquence de base ou "note" principale du son, exprimée en Hz.

**Exemples musicaux** :
- 110 Hz = La grave
- 220 Hz = La médium
- 440 Hz = La standard (diapason)
- 880 Hz = La aigu

**Applications** :
- Détection de la hauteur tonale (pitch)
- Analyse vocale (F0 varie selon le locuteur)
- Détection de fréquence de vibration en industrie

### Centroïde spectral
Le "centre de gravité" du spectre fréquentiel. C'est la fréquence autour de laquelle l'énergie sonore est équilibrée.

**Corrélation perceptuelle** : **Brillance du son**
- Centroïde bas (< 1000 Hz) → son sombre, mat, grave
- Centroïde moyen (1000-3000 Hz) → son équilibré
- Centroïde élevé (> 3000 Hz) → son brillant, clair, aigu

**Formule conceptuelle** : Moyenne des fréquences pondérée par leur magnitude.

**Exemples** :
- Son de contrebasse : centroïde bas (~500 Hz)
- Voix humaine : centroïde moyen (~1500-2500 Hz)
- Cymbale : centroïde élevé (> 5000 Hz)

### Rolloff spectral
La fréquence en dessous de laquelle se concentre 85-95% de l'énergie spectrale totale.

**Signification** :
- Indique la "limite supérieure effective" du contenu fréquentiel
- Distingue les sons avec/sans hautes fréquences
- Rolloff bas → peu de contenu aigu
- Rolloff élevé → présence de hautes fréquences

**Applications** :
- Classification audio (parole vs musique)
- Détection de filtrage passe-bas
- Analyse de qualité audio

### Bande passante spectrale
La dispersion ou "étalement" des fréquences autour du centroïde spectral. C'est l'écart-type du spectre.

**Interprétation** :
- Bande passante faible → son pur, tonal (sifflement, diapason)
- Bande passante élevée → son riche, complexe, bruité (bruit blanc, crash)

**Relation avec le timbre** : Mesure la "richesse harmonique" du son.

---

## Caractéristiques temporelles

### Zero Crossing Rate (ZCR)
Le taux de passage par zéro du signal audio, c'est-à-dire le nombre de fois où le signal traverse la ligne d'amplitude zéro par unité de temps.

**Corrélation** : **Contenu haute fréquence**
- ZCR faible (~0.01-0.05) → son grave, basses fréquences
- ZCR moyen (~0.05-0.15) → son équilibré
- ZCR élevé (> 0.15) → son aigu, hautes fréquences, bruit

**Applications** :
- Distinction parole/musique
- Détection de segments voisés/non-voisés
- Classification de bruits (tonaux vs bruités)

**Exemple visuel** :
Signal basse fréquence (50 Hz) : ~~~~~~ (peu de passages par zéro)
Signal haute fréquence (1000 Hz) : ^v^v^v^v^v (beaucoup de passages par zéro)


---

## Représentations avancées

### MFCCs (Mel-Frequency Cepstral Coefficients)
Une représentation compacte du spectre audio basée sur l'échelle mel (qui imite la perception humaine des fréquences).

**Structure** : Tableau à 2 dimensions `(n_coefficients, n_frames)`
- **n_coefficients** : généralement 13 ou 20 coefficients
- **n_frames** : nombre de fenêtres temporelles analysées

**Signification des coefficients** :
- Coefficient 0 : énergie totale du signal
- Coefficients 1-3 : enveloppe spectrale générale (timbre global)
- Coefficients 4-13 : détails spectraux (nuances du timbre)

**Applications principales** :
- Reconnaissance vocale (standard dans tous les systèmes STT)
- Classification de sons
- Identification de locuteurs
- Détection d'anomalies audio industrielles

**Avantage** : Capture la "signature" unique du timbre d'un son de manière compacte.

---

## Récapitulatif : Que mesure chaque métrique ?

| Métrique | Mesure | Usage principal |
|----------|--------|----------------|
| **Durée** | Temps total | Information de base |
| **Sample Rate** | Résolution temporelle | Qualité audio, fréquence max capturable |
| **RMS** | Niveau sonore moyen | Intensité, dynamique |
| **F0** | Fréquence de base | Hauteur tonale, pitch |
| **Centroïde spectral** | "Centre de gravité" fréquentiel | **Brillance** du son |
| **Rolloff spectral** | Limite haute fréquence | Contenu aigu |
| **Bande passante** | Richesse spectrale | Complexité, timbre |
| **Zero Crossing Rate** | Passages par zéro | **Contenu haute fréquence** |
| **MFCCs** | Empreinte timbre | Classification, reconnaissance |

---

## Interprétation combinée

Pour comprendre pleinement un son, il faut croiser les métriques :

**Exemple 1 : Son de flûte**
- F0 : 440 Hz (note La)
- Centroïde : 2500 Hz (brillant)
- Bande passante : faible (son pur)
- ZCR : moyen

**Exemple 2 : Bruit blanc**
- F0 : non détectable (pas de fondamentale)
- Centroïde : ~8000 Hz (très brillant)
- Bande passante : très élevée (toutes les fréquences)
- ZCR : très élevé

**Exemple 3 : Moteur diesel**
- F0 : 100-200 Hz (vibration de base)
- Centroïde : 1000-2000 Hz (peu brillant)
- Bande passante : élevée (harmoniques + bruit)
- ZCR : faible à moyen

---

## Ressources complémentaires

**Bibliothèques Python** :
- `librosa` : extraction de toutes ces features
- `scipy.signal` : traitement du signal
- `soundfile` : lecture/écriture audio

**Concepts liés** :
- Transformée de Fourier (FFT)
- Spectrogramme
- Analyse temps-fréquence
- Perception psychoacoustique


In [18]:
# Cellule 2 : Fonctions d'interprétation des métriques
"""
═══════════════════════════════════════════════════════════════
Fonctions d'interprétation des caractéristiques audio
═══════════════════════════════════════════════════════════════
"""

def interpret_rms(rms_db_mean: float) -> str:
    """
    Interprète le niveau RMS (Root Mean Square) en dB.
    
    Args:
        rms_db_mean: Niveau RMS moyen en décibels
        
    Returns:
        Description textuelle du niveau sonore
    """
    if rms_db_mean > -10:
        return "→ Son TRÈS FORT (proche de la saturation)"
    elif rms_db_mean > -20:
        return "→ Son FORT (niveau élevé)"
    elif rms_db_mean > -30:
        return "→ Son MODÉRÉ (niveau moyen)"
    elif rms_db_mean > -40:
        return "→ Son FAIBLE (niveau bas)"
    else:
        return "→ Son TRÈS FAIBLE (quasi-silence)"


def interpret_f0(f0_mean: float) -> str:
    """
    Interprète la fréquence fondamentale (F0).
    
    Args:
        f0_mean: Fréquence fondamentale moyenne en Hz
        
    Returns:
        Description de la hauteur tonale
    """
    if f0_mean < 100:
        note = "grave (< Do2)"
    elif f0_mean < 200:
        note = "médium-grave (Do2-Sol2)"
    elif f0_mean < 400:
        note = "médium (Sol2-Sol3)"
    elif f0_mean < 800:
        note = "médium-aigu (Sol3-Sol4)"
    else:
        note = "aigu (> Sol4)"
    return f"→ Fréquence de base {note} - vibration à {f0_mean:.0f} Hz"


def interpret_centroid(centroid_mean: float) -> str:
    """
    Interprète le centroïde spectral (brillance du son).
    
    Args:
        centroid_mean: Centroïde spectral moyen en Hz
        
    Returns:
        Description de la brillance perceptuelle
    """
    if centroid_mean < 1000:
        return "→ Son SOMBRE/MAT (peu brillant, dominé par les basses)"
    elif centroid_mean < 2000:
        return "→ Son PEU BRILLANT (fréquences moyennes dominantes)"
    elif centroid_mean < 3000:
        return "→ Son MOYENNEMENT BRILLANT (équilibré)"
    elif centroid_mean < 5000:
        return "→ Son BRILLANT (présence d'aigus marquée)"
    else:
        return "→ Son TRÈS BRILLANT (dominé par les hautes fréquences)"


def interpret_rolloff(rolloff_mean: float) -> str:
    """
    Interprète le rolloff spectral (85% de l'énergie).
    
    Args:
        rolloff_mean: Fréquence de rolloff moyenne en Hz
        
    Returns:
        Description de la distribution spectrale
    """
    if rolloff_mean < 2000:
        return "→ Contenu principalement BASSES FRÉQUENCES"
    elif rolloff_mean < 4000:
        return "→ Contenu jusqu'aux FRÉQUENCES MOYENNES"
    elif rolloff_mean < 8000:
        return "→ Présence de HAUTES FRÉQUENCES modérée"
    else:
        return "→ Riche en TRÈS HAUTES FRÉQUENCES"


def interpret_bandwidth(bandwidth_mean: float) -> str:
    """
    Interprète la bande passante spectrale (richesse harmonique).
    
    Args:
        bandwidth_mean: Bande passante moyenne en Hz
        
    Returns:
        Description de la complexité spectrale
    """
    if bandwidth_mean < 1000:
        return "→ Son PUR/TONAL (peu de composantes fréquentielles)"
    elif bandwidth_mean < 2500:
        return "→ Son MOYENNEMENT RICHE (quelques harmoniques)"
    elif bandwidth_mean < 4000:
        return "→ Son RICHE (spectre complexe avec harmoniques)"
    else:
        return "→ Son TRÈS COMPLEXE/BRUITÉ (large spectre fréquentiel)"


def interpret_zcr(zcr_mean: float) -> str:
    """
    Interprète le zero crossing rate (contenu haute fréquence).
    
    Args:
        zcr_mean: Taux de passage par zéro moyen
        
    Returns:
        Description du contenu fréquentiel
    """
    if zcr_mean < 0.05:
        return "→ Dominance BASSES FRÉQUENCES (son grave)"
    elif zcr_mean < 0.15:
        return "→ Contenu fréquentiel ÉQUILIBRÉ"
    else:
        return "→ Dominance HAUTES FRÉQUENCES (son aigu/bruité)"


def interpret_mfccs(mfccs_shape: tuple) -> str:
    """
    Interprète la forme des MFCCs (Mel-Frequency Cepstral Coefficients).
    
    Args:
        mfccs_shape: Tuple (n_coefficients, n_frames)
        
    Returns:
        Description de la représentation MFCC
    """
    n_coef, n_frames = mfccs_shape
    duration_analysis = n_frames * 0.023  # ~23ms par frame en moyenne
    return f"→ {n_coef} coefficients sur {n_frames} fenêtres temporelles (~{duration_analysis:.1f}s)"


print("✓ Fonctions d'interprétation chargées")


✓ Fonctions d'interprétation chargées


In [19]:
# Cellule 3 : Fonction d'analyse audio complète
"""
═══════════════════════════════════════════════════════════════
Fonction principale d'analyse audio
═══════════════════════════════════════════════════════════════
"""

def analyze_audio_file(audio_file: Path, verbose: bool = True) -> dict:
    """
    Analyse complète d'un fichier audio avec extraction de features.
    
    Args:
        audio_file: Chemin vers le fichier audio
        verbose: Si True, affiche les résultats détaillés
        
    Returns:
        Dictionnaire contenant toutes les features extraites
        
    Raises:
        FileNotFoundError: Si le fichier n'existe pas
        ValueError: Si le fichier ne peut pas être chargé
    """
    
    # Vérification de l'existence du fichier
    if not audio_file.exists():
        raise FileNotFoundError(f"Fichier introuvable: {audio_file}")
    
    if verbose:
        print("="*60)
        print("ANALYSE AUDIO DÉTAILLÉE")
        print("="*60)
        print(f"\n📁 Fichier: {audio_file.name}\n")
    
    # Chargement du fichier
    y, sr = librosa.load(str(audio_file), sr=None)
    
    # ==== INFORMATIONS TEMPORELLES ====
    duration = librosa.get_duration(y=y, sr=sr)
    
    if verbose:
        print("━"*60)
        print("📊 INFORMATIONS TEMPORELLES")
        print("━"*60)
        print(f"Durée: {duration:.2f} secondes")
        print(f"Taux d'échantillonnage: {sr} Hz")
        
        if sr == 44100:
            print("   → Standard CD audio (qualité professionnelle)")
        elif sr == 48000:
            print("   → Standard production audio/vidéo")
        elif sr == 16000:
            print("   → Qualité téléphonie/reconnaissance vocale")
        
        print(f"Nombre d'échantillons: {len(y):,}")
    
    # ==== INTENSITÉ ====
    rms = librosa.feature.rms(y=y)[0]
    rms_db = librosa.amplitude_to_db(rms)
    rms_mean = np.mean(rms_db)
    rms_std = np.std(rms_db)
    
    if verbose:
        print("\n" + "━"*60)
        print("🔊 INTENSITÉ SONORE")
        print("━"*60)
        print(f"Intensité RMS moyenne: {rms_mean:.2f} dB")
        print(f"   {interpret_rms(rms_mean)}")
        print(f"Écart-type RMS: {rms_std:.2f} dB")
        
        if rms_std > 10:
            print(f"   → Dynamique IMPORTANTE (grandes variations d'intensité)")
        elif rms_std > 5:
            print(f"   → Dynamique MODÉRÉE")
        else:
            print(f"   → Dynamique FAIBLE (son stable)")
    
    # ==== FRÉQUENCES ====
    f0 = librosa.yin(y, fmin=50, fmax=400)
    spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    zero_crossing_rate = librosa.feature.zero_crossing_rate(y)[0]
    
    centroid_mean = np.mean(spectral_centroids)
    rolloff_mean = np.mean(spectral_rolloff)
    bandwidth_mean = np.mean(spectral_bandwidth)
    zcr_mean = np.mean(zero_crossing_rate)
    
    if verbose:
        print("\n" + "━"*60)
        print("🎵 ANALYSE FRÉQUENTIELLE")
        print("━"*60)
        
        if np.any(f0 > 0):
            f0_mean = np.mean(f0[f0>0])
            print(f"F0 moyenne (fréquence fondamentale): {f0_mean:.2f} Hz")
            print(f"   {interpret_f0(f0_mean)}")
        else:
            print("F0: Non détectable (pas de fondamentale claire)")
            f0_mean = None
        
        print(f"\nCentroïde spectral moyen: {centroid_mean:.2f} Hz")
        print(f"   {interpret_centroid(centroid_mean)}")
        print(f"   💡 Le centroïde mesure la BRILLANCE perçue du son")
        
        print(f"\nRolloff spectral moyen: {rolloff_mean:.2f} Hz")
        print(f"   {interpret_rolloff(rolloff_mean)}")
        print(f"   💡 85% de l'énergie est sous {rolloff_mean:.0f} Hz")
        
        print(f"\nBande passante moyenne: {bandwidth_mean:.2f} Hz")
        print(f"   {interpret_bandwidth(bandwidth_mean)}")
        print(f"   💡 Mesure la richesse/complexité harmonique")
        
        print(f"\nZero crossing rate moyen: {zcr_mean:.4f}")
        print(f"   {interpret_zcr(zcr_mean)}")
        print(f"   💡 Indique le contenu en hautes fréquences")
    
    # ==== MFCCs ====
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    
    if verbose:
        print("\n" + "━"*60)
        print("🔬 REPRÉSENTATION AVANCÉE")
        print("━"*60)
        print(f"MFCCs shape: {mfccs.shape}")
        print(f"   {interpret_mfccs(mfccs.shape)}")
        print(f"   💡 'Empreinte digitale' du timbre pour classification")
    
    # ==== RÉSUMÉ ====
    if verbose:
        print("\n" + "="*60)
        print("📋 RÉSUMÉ INTERPRÉTATIF")
        print("="*60)
        print(f"\nCe fichier audio de {duration:.0f}s est un son:")
        
        if rms_mean > -20:
            print("  • FORT en intensité")
        elif rms_mean > -30:
            print("  • MODÉRÉ en intensité")
        else:
            print("  • FAIBLE en intensité")
        
        if centroid_mean < 2000:
            print(f"  • PEU BRILLANT (sombre)")
        elif centroid_mean < 4000:
            print(f"  • MOYENNEMENT BRILLANT")
        else:
            print(f"  • TRÈS BRILLANT (clair)")
        
        if bandwidth_mean > 3000:
            print(f"  • RICHE en harmoniques (spectre complexe)")
        else:
            print(f"  • Spectre relativement simple")
        
        if zcr_mean < 0.05:
            print(f"  • Dominé par les BASSES FRÉQUENCES")
        elif zcr_mean > 0.15:
            print(f"  • Dominé par les HAUTES FRÉQUENCES")
        else:
            print(f"  • Équilibré fréquentiellement")
        
        print("\n" + "="*60)
    
    # Retourner un dictionnaire avec toutes les features
    return {
        'filename': audio_file.name,
        'duration': duration,
        'sample_rate': sr,
        'n_samples': len(y),
        'rms_mean_db': rms_mean,
        'rms_std_db': rms_std,
        'f0_mean': f0_mean if np.any(f0 > 0) else None,
        'spectral_centroid_mean': centroid_mean,
        'spectral_rolloff_mean': rolloff_mean,
        'spectral_bandwidth_mean': bandwidth_mean,
        'zero_crossing_rate_mean': zcr_mean,
        'mfccs_shape': mfccs.shape,
        'signal': y,
        'sr': sr
    }


print("✓ Fonction d'analyse chargée")


✓ Fonction d'analyse chargée


## 📊 Démonstration : Analyse d'un fichier audio

Analyse d'un son industriel (pont roulant).

In [20]:
# Cellule 4 : Exemple d'analyse
"""
Analyse d'un fichier audio industriel
"""

# Sélectionner le fichier à analyser
audio_file = AUDIO_INDUSTRIAL_DIR / "AV-1-S-OUT-401.wav"

# Lancer l'analyse
features = analyze_audio_file(audio_file, verbose=True)

# Les features sont maintenant disponibles dans le dictionnaire 'features'
print(f"\n✓ Features extraites et stockées dans la variable 'features'")


ANALYSE AUDIO DÉTAILLÉE

📁 Fichier: AV-1-S-OUT-401.wav

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 INFORMATIONS TEMPORELLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Durée: 46.08 secondes
Taux d'échantillonnage: 44100 Hz
   → Standard CD audio (qualité professionnelle)
Nombre d'échantillons: 2,032,128

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔊 INTENSITÉ SONORE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Intensité RMS moyenne: -26.38 dB
   → Son MODÉRÉ (niveau moyen)
Écart-type RMS: 21.81 dB
   → Dynamique IMPORTANTE (grandes variations d'intensité)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎵 ANALYSE FRÉQUENTIELLE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
F0 moyenne (fréquence fondamentale): 139.96 Hz
   → Fréquence de base médium-grave (Do2-Sol2) - vibration à 140 Hz

Centroïde spectral moyen: 2120.46 Hz
   → Son MOYENNEMENT BRILLANT (équilibré)
   💡 Le centroïde mesure l

## 🔉 Réduction de volume audio

Ajustement du niveau sonore d'un fichier audio en utilisant une conversion dB → amplitude.


In [21]:
# Cellule 5 : Fonction de réduction de volume
"""
═══════════════════════════════════════════════════════════════
Fonction d'ajustement du volume audio
═══════════════════════════════════════════════════════════════
"""

def adjust_audio_volume(input_file: Path, output_file: Path, reduction_db: float = -10):
    """
    Ajuste le volume d'un fichier audio.
    
    Args:
        input_file: Fichier audio d'entrée
        output_file: Fichier audio de sortie
        reduction_db: Réduction en dB (négatif pour baisser, positif pour augmenter)
        
    Returns:
        Tuple (signal ajusté, sample rate)
        
    Example:
        >>> adjust_audio_volume(input_file, output_file, reduction_db=-10)
        RMS actuel: -26.38 dB
        RMS après réduction: -36.38 dB
        ✓ Fichier sauvegardé: output.wav
    """
    
    print(f"\n🔊 AJUSTEMENT DU VOLUME")
    print("="*60)
    
    # Charger l'audio
    y, sr = librosa.load(str(input_file), sr=None)
    
    # Analyser le niveau actuel
    rms_current = np.mean(librosa.amplitude_to_db(librosa.feature.rms(y=y)[0]))
    print(f"📊 RMS actuel: {rms_current:.2f} dB")
    
    # Appliquer la réduction
    gain = 10 ** (reduction_db / 20)  # Conversion dB vers amplitude
    y_adjusted = y * gain
    
    # Vérifier le nouveau niveau
    rms_new = np.mean(librosa.amplitude_to_db(librosa.feature.rms(y=y_adjusted)[0]))
    print(f"📊 RMS après ajustement: {rms_new:.2f} dB")
    print(f"   Différence appliquée: {rms_new - rms_current:.2f} dB")
    
    # Sauvegarder
    sf.write(str(output_file), y_adjusted, sr)
    print(f"✓ Fichier sauvegardé: {output_file}\n")
    
    return y_adjusted, sr


print("✓ Fonction d'ajustement de volume chargée")


✓ Fonction d'ajustement de volume chargée


In [22]:
# Cellule 6 : Exemple de réduction de volume
"""
Exemple : Réduire le volume d'un fichier audio de -10 dB
"""

input_file = AUDIO_INDUSTRIAL_DIR / "AV-1-S-OUT-401.wav"
output_file = AUDIO_INDUSTRIAL_DIR / "AV-1-S-OUT-401_reduced.wav"

# Réduire de 10 dB
y_reduced, sr = adjust_audio_volume(input_file, output_file, reduction_db=-10)



🔊 AJUSTEMENT DU VOLUME
📊 RMS actuel: -26.38 dB
📊 RMS après ajustement: -36.38 dB
   Différence appliquée: -10.00 dB
✓ Fichier sauvegardé: /Users/julienrm/Workspace/Lab_IA/memoiredesterritoires/data/audio/background_sounds/Pont roulant/AV-1-S-OUT-401_reduced.wav



## 🎙️ Mixage voix + bruit de fond

Superposition d'un bruit industriel sur une voix avec contrôle du rapport signal/bruit (SNR).

**Signal-to-Noise Ratio (SNR)** :
- **SNR 20+ dB** : Voix très claire, bruit discret
- **SNR 10-20 dB** : Voix claire, bruit perceptible (recommandé)
- **SNR 0-10 dB** : Voix difficile, bruit important
- **SNR < 0 dB** : Voix noyée dans le bruit


In [23]:
# Cellule 7 : Fonction de mixage voix + bruit
"""
═══════════════════════════════════════════════════════════════
Fonction de mixage voix + bruit avec contrôle SNR
═══════════════════════════════════════════════════════════════
"""

def mix_voice_with_noise(
    voice_file: Path,
    noise_file: Path,
    output_file: Path,
    snr_db: float = 15,
    start_time: float = 0,
    noise_duration: float = None,
    noise_start_offset: float = 0
) -> tuple:
    """
    Superpose un bruit sur une voix avec un contrôle précis du SNR.
    
    Args:
        voice_file: Fichier audio de la voix
        noise_file: Fichier audio du bruit (industriel, ambiance)
        output_file: Fichier de sortie
        snr_db: Rapport signal/bruit en dB (15 = voix 15dB plus forte que le bruit)
        start_time: Moment où le bruit commence dans la VOIX (en secondes)
        noise_duration: Durée du bruit (None = jusqu'à la fin de la voix)
        noise_start_offset: Début de lecture dans le fichier BRUIT (en secondes)
                           Utile pour éviter les silences au début du bruit
        
    Returns:
        Tuple (signal mixé, sample rate)
        
    Raises:
        FileNotFoundError: Si un fichier n'existe pas
        ValueError: Si l'offset du bruit est invalide
        
    Example:
        >>> mixed, sr = mix_voice_with_noise(
        ...     voice_file=Path("voice.mp3"),
        ...     noise_file=Path("industrial.wav"),
        ...     output_file=Path("mixed.wav"),
        ...     snr_db=20,
        ...     start_time=5.0,
        ...     noise_duration=10.0,
        ...     noise_start_offset=3.0
        ... )
    """
    
    print("="*60)
    print("🎙️ MIXAGE VOIX + BRUIT DE FOND")
    print("="*60)
    
    # Vérification de l'existence des fichiers
    if not voice_file.exists():
        raise FileNotFoundError(f"Fichier voix introuvable: {voice_file}")
    if not noise_file.exists():
        raise FileNotFoundError(f"Fichier bruit introuvable: {noise_file}")
    
    # 1. CHARGER LES FICHIERS
    print("\n📂 Chargement des fichiers...")
    voice, sr_voice = librosa.load(str(voice_file), sr=None)
    noise, sr_noise = librosa.load(str(noise_file), sr=None)
    
    print(f"   Voix: {len(voice)/sr_voice:.2f}s @ {sr_voice}Hz")
    print(f"   Bruit: {len(noise)/sr_noise:.2f}s @ {sr_noise}Hz")
    
    # 2. RESAMPLER SI NÉCESSAIRE
    if sr_voice != sr_noise:
        print(f"\n⚙️ Resampling du bruit: {sr_noise}Hz → {sr_voice}Hz")
        noise = librosa.resample(noise, orig_sr=sr_noise, target_sr=sr_voice)
        sr_noise = sr_voice
    
    sr = sr_voice
    
    # 3. EXTRAIRE LA PARTIE DU BRUIT À PARTIR DE L'OFFSET
    noise_offset_sample = int(noise_start_offset * sr)
    
    if noise_offset_sample >= len(noise):
        raise ValueError(
            f"⚠️ Offset du bruit ({noise_start_offset}s) dépasse "
            f"la durée du fichier ({len(noise)/sr:.2f}s)"
        )
    
    if noise_offset_sample > 0:
        noise = noise[noise_offset_sample:]
        print(f"\n✂️ Bruit extrait à partir de {noise_start_offset:.2f}s")
        print(f"   Nouvelle durée disponible: {len(noise)/sr:.2f}s")
    
    # 4. CALCULER LES NIVEAUX RMS
    rms_voice = np.sqrt(np.mean(voice**2))
    rms_noise = np.sqrt(np.mean(noise**2))
    
    print(f"\n📊 Niveaux RMS originaux:")
    print(f"   Voix: {20*np.log10(rms_voice + 1e-10):.2f} dB")
    print(f"   Bruit: {20*np.log10(rms_noise + 1e-10):.2f} dB")
    
    # 5. AJUSTER LE NIVEAU DU BRUIT SELON LE SNR SOUHAITÉ
    # Formule: SNR = 20*log10(RMS_voix / RMS_bruit)
    # Donc: RMS_bruit_cible = RMS_voix / 10^(SNR/20)
    
    target_rms_noise = rms_voice / (10 ** (snr_db / 20))
    noise_gain = target_rms_noise / (rms_noise + 1e-10)
    noise_adjusted = noise * noise_gain
    
    print(f"\n🎚️ Ajustement du bruit pour SNR = {snr_db} dB:")
    print(f"   Gain appliqué au bruit: {20*np.log10(noise_gain):.2f} dB")
    print(f"   Nouveau niveau bruit: {20*np.log10(target_rms_noise):.2f} dB")
    
    # 6. PRÉPARER LE BRUIT À SUPERPOSER
    start_sample = int(start_time * sr)
    
    # Déterminer la durée du bruit
    if noise_duration is None:
        available_length = len(voice) - start_sample
    else:
        available_length = int(noise_duration * sr)
    
    # Gérer le cas où le bruit est plus court que nécessaire (boucler)
    if len(noise_adjusted) < available_length:
        print(f"\n🔁 Bruit trop court ({len(noise_adjusted)/sr:.2f}s), création d'une boucle...")
        repeats = int(np.ceil(available_length / len(noise_adjusted)))
        noise_adjusted = np.tile(noise_adjusted, repeats)
        print(f"   Répété {repeats} fois pour obtenir {len(noise_adjusted)/sr:.2f}s")
    
    # Couper le bruit à la longueur voulue
    noise_segment = noise_adjusted[:available_length]
    
    # 7. CRÉER LE SIGNAL MIXÉ
    mixed = voice.copy()
    
    # Ajouter le bruit à partir de start_sample
    end_sample = min(start_sample + len(noise_segment), len(mixed))
    actual_noise_length = end_sample - start_sample
    mixed[start_sample:end_sample] += noise_segment[:actual_noise_length]
    
    print(f"\n🔀 Mixage:")
    print(f"   Bruit source: {noise_start_offset:.2f}s → "
          f"{noise_start_offset + actual_noise_length/sr:.2f}s (fichier bruit)")
    print(f"   Position dans voix: {start_time:.2f}s → {(end_sample/sr):.2f}s")
    print(f"   Durée du bruit ajouté: {actual_noise_length/sr:.2f}s")
    
    # 8. NORMALISER SI NÉCESSAIRE (éviter la saturation)
    max_amplitude = np.max(np.abs(mixed))
    if max_amplitude > 0.99:
        print(f"\n⚠️ Saturation détectée ({max_amplitude:.3f}), normalisation appliquée")
        mixed = mixed / max_amplitude * 0.95
    
    # 9. VÉRIFIER LE SNR FINAL
    voice_segment = voice[start_sample:end_sample]
    rms_voice_seg = np.sqrt(np.mean(voice_segment**2))
    rms_noise_seg = np.sqrt(np.mean(noise_segment[:actual_noise_length]**2))
    actual_snr = 20 * np.log10(rms_voice_seg / (rms_noise_seg + 1e-10))
    
    print(f"\n✓ SNR final mesuré: {actual_snr:.2f} dB")
    
    if actual_snr >= 20:
        print("   → Excellente intelligibilité (voix très claire)")
    elif actual_snr >= 10:
        print("   → Bonne intelligibilité (léger bruit de fond)")
    elif actual_snr >= 0:
        print("   → Intelligibilité acceptable (bruit perceptible)")
    else:
        print("   → ⚠️ Intelligibilité difficile (bruit fort)")
    
    # 10. SAUVEGARDER
    output_file.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(output_file), mixed, sr)
    print(f"\n💾 Fichier sauvegardé: {output_file}")
    print("="*60)
    
    return mixed, sr


print("✓ Fonction de mixage chargée")


✓ Fonction de mixage chargée


In [24]:
# Cellule 8 : Exemple de mixage
"""
Exemple : Mixer une voix avec un bruit industriel
"""

# Configuration du mixage
CONFIG = {
    'voice_file': AUDIO_VOICE_DIR / "ElevenLabs_Spuds_Oxley.mp3",
    'noise_file': AUDIO_INDUSTRIAL_DIR / "AV-1-S-OUT-401.wav",
    'output_file': OUTPUT_DIR / "mixage_SNR20.wav",
    'snr_db': 20,  # Voix 20 dB plus forte que le bruit
    'start_time': 5.0,  # Le bruit apparaît à 5s de la voix
    'noise_duration': 5.0,  # Le bruit dure 5 secondes
    'noise_start_offset': 5.0  # Commence à lire le bruit à 5s (évite le silence)
}

# Lancer le mixage
mixed, sr = mix_voice_with_noise(**CONFIG)

print(f"\n🎉 Mixage terminé ! Le fichier est disponible dans : {CONFIG['output_file']}")


🎙️ MIXAGE VOIX + BRUIT DE FOND


FileNotFoundError: Fichier voix introuvable: /Users/julienrm/Workspace/Lab_IA/memoiredesterritoires/data/Speech Final/ElevenLabs_Spuds_Oxley.mp3

# Guide des courbes de fondu audio (Fade In / Fade Out)

## Introduction

Un **fondu audio** (fade) est une transition progressive entre deux niveaux sonores, généralement :
- **Fade In** : passage du silence au volume nominal (0 → 1)
- **Fade Out** : passage du volume nominal au silence (1 → 0)
- **Crossfade** : transition entre deux sons (son A disparaît pendant que son B apparaît)

**Pourquoi différentes courbes ?** Notre oreille perçoit le volume de façon **logarithmique**, pas linéaire. Un changement de 50% d'amplitude ne donne pas une impression de 50% de volume ! Chaque courbe compense différemment cette perception.

---

## Les 5 types de courbes de fondu

### 1. Courbe Linéaire

**Formule mathématique** : `f(t) = t` où t ∈ [0, 1]

**Représentation visuelle** :
``` Amplitud
1.0 | --- | / 0.5 | / | / 0.0 |/_ 0s → 1s
```

**Caractéristiques** :
- Pente constante en amplitude
- Simple mathématiquement
- Transition perçue comme **accélérée au milieu**

**Effet perceptuel** :
- ❌ Pas naturel pour l'oreille humaine
- Le volume semble augmenter brusquement vers 0.5s
- Peut créer une impression de "saut" de volume

**Quand l'utiliser** :
- Effets spéciaux rapides (< 0.2s)
- Transitions volontairement brusques
- **Non recommandé pour de la musique**

---

### 2. Courbe Exponentielle (Quadratique)

**Formule mathématique** : `f(t) = t²`

**Représentation visuelle** :
``` Amplitude
1.0 | ___
| /
0.5 | /
| ____/
0.0 |/________
0s → 1s
```


**Caractéristiques** :
- Début **lent**, accélération progressive
- Correspond à l'inverse de la perception logarithmique
- Transition perçue comme **régulière**

**Effet perceptuel** :
- ✅ Naturel pour l'oreille
- Le volume monte/descend de façon constante perceptuellement
- Doux au début, décisif à la fin

**Quand l'utiliser** :
- **⭐ FADE OUT (recommandé)** : disparition naturelle
- Fins de morceaux musicaux
- Sons qui s'éloignent
- Voix qui s'estompe

**Pourquoi pour fade out ?** Le son reste audible longtemps (début lent), puis disparaît rapidement (évite une traînée désagréable).

---

### 3. Courbe Logarithmique (Racine carrée)

**Formule mathématique** : `f(t) = √t`

**Représentation visuelle** :
Amplitude
1.0 | _______
| /
0.5 | /
| /
0.0 |/_________
0s → 1s


**Caractéristiques** :
- Début **rapide**, ralentissement progressif
- Inverse de la courbe exponentielle
- Monte vite puis se stabilise

**Effet perceptuel** :
- ✅ Naturel pour l'oreille
- Le son apparaît rapidement et clairement
- Montée douce vers le volume final

**Quand l'utiliser** :
- **⭐ FADE IN (recommandé)** : apparition naturelle
- Débuts de morceaux musicaux
- Sons qui s'approchent
- Entrée d'un instrument ou d'une voix

**Pourquoi pour fade in ?** Le son est rapidement identifiable (début rapide), puis monte doucement jusqu'au niveau final (évite un pic brutal).

---

### 4. Courbe Equal Power (Puissance constante)

**Formule mathématique** :
- Fade in : `f(t) = sin(t × π/2)`
- Fade out : `f(t) = cos(t × π/2)`

**Représentation visuelle** :
Amplitude
1.0 | _____
| /
0.5 | /
| /
0.0 |/________
0s → 1s


**Caractéristiques** :
- Basée sur les fonctions trigonométriques
- **Propriété magique** : `sin²(t) + cos²(t) = 1`
- La puissance totale reste constante lors d'un crossfade

**Effet perceptuel** :
- ✅ Volume perçu constant pendant la transition
- Pas de "trou" de volume au milieu
- Transition imperceptible entre deux sons

**Quand l'utiliser** :
- **⭐ CROSSFADE (recommandé)** : transition entre 2 sons
- DJ mix entre deux morceaux
- Changements de scène au cinéma
- Boucles audio (loop sans coupure)

**Pourquoi "equal power" ?** Quand deux signaux se croisent avec cette courbe, la somme de leurs puissances = 1 à tout instant. Résultat : volume constant perçu.

---

### 5. Courbe Sigmoïde (en S)

**Formule mathématique** : `f(t) = 1 / (1 + e^(-k(t-0.5)))` où k contrôle la raideur

**Représentation visuelle** :
Amplitude
1.0 | ___
| _/
0.5 | /
| /
0.0 |__/
0s → 1s


**Caractéristiques** :
- Forme de "S" symétrique
- Extrémités **très douces** (presque plates)
- Centre de la transition accéléré

**Effet perceptuel** :
- ✅ La plus douce de toutes les courbes
- Début et fin imperceptibles
- Transition ultra-smooth, cinématographique

**Quand l'utiliser** :
- **⭐ FONDUS LONGS** (> 2 secondes)
- Ambiances cinématographiques
- Musique classique, ambient
- Transitions atmosphériques

**Pourquoi sigmoïde ?** Les extrémités plates évitent toute brusquerie. Idéal pour des transitions longues et subtiles.

---

## Comparaison visuelle des courbes

Amplitude
1.0 |
| ┌──── Sigmoïde (S)
| ╱
| ╱ ┌─── Exponentielle (x²)
0.5 | ╱ ╱
|╱ ╱ ┌─── Logarithmique (√x)
╱ ╱ ╱
0.0 ╱──╱──╱──── Linéaire (x)
└──────────────────→
0s Temps 1s


---

## Tableau récapitulatif

| Courbe | Formule | Début | Milieu | Fin | Meilleur usage | Notes |
|--------|---------|-------|---------|-----|----------------|-------|
| **Linéaire** | `t` | Moyen | Rapide ⚡ | Moyen | Effets courts | Perçu comme brusque |
| **Exponentielle** | `t²` | Lent 🐌 | Progressif | Rapide ⚡ | **Fade OUT** ⭐ | Standard industrie |
| **Logarithmique** | `√t` | Rapide ⚡ | Progressif | Lent 🐌 | **Fade IN** ⭐ | Standard industrie |
| **Equal Power** | `sin/cos` | Équilibré | Équilibré | Équilibré | **Crossfade** ⭐ | Volume constant |
| **Sigmoïde** | `1/(1+e^-x)` | Très doux 🌙 | Rapide ⚡ | Très doux 🌙 | Fondus longs, cinéma | Ultra-smooth |

---

## Recommandations pratiques

### Pour un son qui apparaît (Fade In)

```python
fade_in_duration = 1.0        # 1 seconde
fade_type = 'logarithmic'     # ⭐ Recommandé
```
**Résultat** : Le son devient rapidement audible (on l'identifie), puis monte doucement jusqu'au volume final (évite un pic brutal).

### Pour un son qui disparaît (Fade Out)

```python
fade_out_duration = 1.0       # 1 seconde
fade_type = 'exponential'     # ⭐ Recommandé
```

**Résultat** : Le son reste audible longtemps (on l'entend s'estomper), puis disparaît rapidement à la fin (évite une traînée).

### Pour une transition entre deux sons (Crossfade)

```python
crossfade_duration = 2.0      # 2 secondes
fade_type = 'equal_power'     # ⭐ Recommandé
```

**Résultat** : Pas de baisse de volume perçue pendant la transition. Le son A s'efface pendant que le son B apparaît, avec un volume total constant.

### Pour un effet cinématographique
```python
fade_in_duration = 3.0        # 3 secondes
fade_out_duration = 3.0
fade_type = 'sigmoid'         # ⭐ Ultra-smooth
```

**Résultat** : Transition très progressive et imperceptible. Parfait pour des ambiances, des nappes sonores, de la musique atmosphérique.

## La règle d'or : LOG IN, EXP OUT
La combinaison logarithmique pour fade in + exponentielle pour fade out est devenue le standard de l'industrie audio car elle compense parfaitement la perception logarithmique de l'oreille humaine.

```python
Volume perçu constant = courbe compensée
      ↓
Fade IN  : √t  (logarithmique)
Fade OUT : t²  (exponentielle)
```

C'est pourquoi tous les DAW (Digital Audio Workstations) professionnels utilisent ces courbes par défaut.

---

## Durées recommandées selon le contexte

| Contexte            | Fade In  | Fade Out | Type        |
| ------------------- | -------- | -------- | ----------- |
| Podcast/Voix        | 0.5-1s   | 0.5-1s   | Log / Exp   |
| Musique pop/rock    | 0.5-2s   | 1-3s     | Log / Exp   |
| Musique classique   | 2-4s     | 3-5s     | Sigmoid     |
| Sound design/SFX    | 0.1-0.5s | 0.2-0.5s | Linear/Log  |
| Ambiance/Drone      | 3-10s    | 3-10s    | Sigmoid     |
| Transition DJ       | 2-8s     | 2-8s     | Equal Power |
| Cinéma/documentaire | 1-3s     | 1-3s     | Sigmoid     |

## Erreurs fréquentes à éviter
❌ Utiliser une courbe linéaire pour de la musique
→ Résultat : transition perçue comme brusque et non professionnelle

❌ Fade trop court pour un son qui dure longtemps
→ Résultat : effet "pop" audible au début/fin

❌ Fade trop long pour un son court
→ Résultat : le son n'atteint jamais son volume nominal

❌ Utiliser exponential pour fade in
→ Résultat : le son met trop de temps à devenir audible

❌ Utiliser logarithmic pour fade out
→ Résultat : le son s'éternise en fin de fondu

---

# Conclusion
Les courbes de fondu ne sont pas qu'un détail technique : elles influencent directement la qualité perceptuelle de vos mixages audio. Utiliser la bonne courbe au bon moment transforme un mixage amateur en production professionnelle.

## Retenez la règle d'or :
- ✨ Fade IN → Logarithmique
- ✨ Fade OUT → Exponentielle
- ✨ Crossfade → Equal Power

## 📝 Notes et prochaines étapes

**Ce notebook permet de** :  
✅ Analyser des fichiers audio industriels (features spectrales et temporelles)  
✅ Ajuster le volume d'un audio (réduction/augmentation en dB)  
✅ Mixer une voix avec un bruit de fond (contrôle SNR)

**Améliorations futures possibles** :
- [ ] Analyse batch de plusieurs fichiers
- [ ] Visualisation des spectrogrammes
- [ ] Export des features en CSV/JSON
- [ ] Détection d'événements sonores (onset detection)
- [ ] Filtrage fréquentiel (passe-bas, passe-haut)


In [ ]:
# Cellule 9 : Mixage multi-bruits avec fondus automatiques
"""
═══════════════════════════════════════════════════════════════
Mixage de plusieurs bruits industriels sur une voix
avec application automatique de la règle d'or des fondus
═══════════════════════════════════════════════════════════════
"""

import librosa
import soundfile as sf
import numpy as np
from pathlib import Path


def create_fade_curve(length: int, fade_type: str = 'linear') -> np.ndarray:
    """
    Crée une courbe de fondu.
    
    Args:
        length: Nombre d'échantillons du fondu
        fade_type: Type de courbe
        
    Returns:
        Array numpy de la courbe (0 à 1)
    """
    t = np.linspace(0, 1, length)
    
    if fade_type == 'linear':
        return t
    elif fade_type == 'exponential':
        return t ** 2
    elif fade_type == 'logarithmic':
        return np.sqrt(t)
    elif fade_type == 'equal_power':
        return np.sin(t * np.pi / 2)
    elif fade_type == 'sigmoid':
        return 1 / (1 + np.exp(-12 * (t - 0.5)))
    else:
        raise ValueError(f"Type de fade inconnu: {fade_type}")


def add_noise_with_fade(
    voice_signal: np.ndarray,
    noise_signal: np.ndarray,
    sr: int,
    start_time: float,
    duration: float,
    snr_db: float,
    noise_offset: float = 0,
    fade_in: float = 0.5,
    fade_out: float = 0.5
) -> np.ndarray:
    """
    Ajoute un bruit à un signal vocal avec fondus automatiques (règle d'or).
    
    Args:
        voice_signal: Signal vocal (array numpy)
        noise_signal: Signal de bruit (array numpy)
        sr: Sample rate
        start_time: Moment où le bruit commence (secondes)
        duration: Durée du bruit (secondes)
        snr_db: Rapport signal/bruit (dB)
        noise_offset: Décalage dans le fichier bruit (secondes)
        fade_in: Durée du fade in (secondes)
        fade_out: Durée du fade out (secondes)
        
    Returns:
        Signal vocal avec bruit ajouté
    """
    
    # 1. EXTRAIRE LA PARTIE DU BRUIT À PARTIR DE L'OFFSET
    noise_offset_sample = int(noise_offset * sr)
    if noise_offset_sample >= len(noise_signal):
        print(f"  ⚠️ Offset trop grand, reset à 0")
        noise_offset_sample = 0
    
    noise = noise_signal[noise_offset_sample:]
    
    # 2. CALCULER LE NIVEAU RMS ET AJUSTER LE SNR
    rms_voice = np.sqrt(np.mean(voice_signal**2))
    rms_noise = np.sqrt(np.mean(noise**2))
    
    # Formule SNR : SNR = 20*log10(RMS_voice / RMS_noise)
    target_rms_noise = rms_voice / (10 ** (snr_db / 20))
    noise_gain = target_rms_noise / (rms_noise + 1e-10)
    noise_adjusted = noise * noise_gain
    
    # 3. PRÉPARER LA DURÉE DU BRUIT
    start_sample = int(start_time * sr)
    duration_samples = int(duration * sr)
    
    # Boucler le bruit si trop court
    if len(noise_adjusted) < duration_samples:
        repeats = int(np.ceil(duration_samples / len(noise_adjusted)))
        noise_adjusted = np.tile(noise_adjusted, repeats)
    
    # Couper à la bonne longueur
    noise_segment = noise_adjusted[:duration_samples]
    
    # 4. APPLIQUER LES FONDUS (RÈGLE D'OR)
    fade_in_samples = int(fade_in * sr)
    fade_out_samples = int(fade_out * sr)
    
    # Vérifier que les fondus ne sont pas trop longs
    total_fade = fade_in_samples + fade_out_samples
    if total_fade >= len(noise_segment):
        # Ajuster automatiquement
        fade_in_samples = len(noise_segment) // 4
        fade_out_samples = len(noise_segment) // 4
    
    # FADE IN avec courbe LOGARITHMIQUE (règle d'or ✨)
    if fade_in_samples > 0:
        fade_in_curve = create_fade_curve(fade_in_samples, 'logarithmic')
        noise_segment[:fade_in_samples] *= fade_in_curve
    
    # FADE OUT avec courbe EXPONENTIELLE (règle d'or ✨)
    if fade_out_samples > 0:
        fade_out_curve = create_fade_curve(fade_out_samples, 'exponential')
        # Inverser pour fade out (1 → 0)
        fade_out_curve = fade_out_curve[::-1]
        noise_segment[-fade_out_samples:] *= fade_out_curve
    
    # 5. AJOUTER LE BRUIT AU SIGNAL VOCAL
    end_sample = min(start_sample + len(noise_segment), len(voice_signal))
    actual_length = end_sample - start_sample
    
    # Créer une copie pour ne pas modifier l'original
    mixed = voice_signal.copy()
    mixed[start_sample:end_sample] += noise_segment[:actual_length]
    
    return mixed


def mix_multiple_industrial_noises(
    voice_file: Path,
    noise_configs: list,
    output_file: Path
) -> tuple:
    """
    Mixe plusieurs bruits industriels sur une voix avec fondus automatiques.
    
    Args:
        voice_file: Fichier audio de la voix
        noise_configs: Liste de dictionnaires de configuration pour chaque bruit
                      Format: {
                          'file': Path,
                          'name': str,
                          'start_time': float,
                          'duration': float,
                          'snr_db': float,
                          'offset': float,
                          'fade_in': float,
                          'fade_out': float
                      }
        output_file: Fichier de sortie
        
    Returns:
        Tuple (signal mixé, sample rate)
        
    Example:
        >>> configs = [
        ...     {'file': Path('chalumeau.wav'), 'name': 'Chalumeau', 
        ...      'start_time': 5.0, 'duration': 3.0, 'snr_db': 15, ...},
        ...     {'file': Path('meule.wav'), 'name': 'Meule',
        ...      'start_time': 10.0, 'duration': 4.0, 'snr_db': 18, ...}
        ... ]
        >>> mix_multiple_industrial_noises(voice_file, configs, output_file)
    """
    
    print("="*70)
    print("🎙️ MIXAGE MULTI-BRUITS INDUSTRIELS AVEC FONDUS AUTOMATIQUES")
    print("="*70)
    print(f"\n🎯 Règle d'or appliquée : FADE IN = Logarithmique | FADE OUT = Exponentielle")
    
    # ÉTAPE 1 : CHARGER LA VOIX
    print(f"\n📂 Chargement de la voix...")
    if not voice_file.exists():
        raise FileNotFoundError(f"Fichier voix introuvable: {voice_file}")
    
    voice, sr = librosa.load(str(voice_file), sr=None)
    voice_duration = len(voice) / sr
    
    print(f"   ✓ Voix chargée : {voice_duration:.2f}s @ {sr}Hz")
    print(f"   Nombre de bruits à mixer : {len(noise_configs)}")
    
    # ÉTAPE 2 : CHARGER TOUS LES BRUITS
    print(f"\n📦 Chargement des bruits industriels...")
    noise_signals = {}
    
    for i, config in enumerate(noise_configs, 1):
        noise_file = config['file']
        noise_name = config['name']
        
        if not noise_file.exists():
            print(f"   ⚠️ [{i}/{len(noise_configs)}] {noise_name} : FICHIER INTROUVABLE, ignoré")
            continue
        
        # Charger le bruit
        noise, sr_noise = librosa.load(str(noise_file), sr=None)
        
        # Resampler si nécessaire
        if sr_noise != sr:
            noise = librosa.resample(noise, orig_sr=sr_noise, target_sr=sr)
        
        noise_signals[noise_name] = noise
        print(f"   ✓ [{i}/{len(noise_configs)}] {noise_name} : {len(noise)/sr:.2f}s")
    
    # ÉTAPE 3 : MIXER PROGRESSIVEMENT CHAQUE BRUIT
    print(f"\n🔊 Mixage des bruits avec fondus...")
    print("─"*70)
    
    mixed = voice.copy()
    
    for i, config in enumerate(noise_configs, 1):
        noise_name = config['name']
        
        # Vérifier que le bruit a été chargé
        if noise_name not in noise_signals:
            continue
        
        # Paramètres du bruit
        start_time = config.get('start_time', 0)
        duration = config.get('duration', 5.0)
        snr_db = config.get('snr_db', 15)
        offset = config.get('offset', 0)
        fade_in = config.get('fade_in', 0.5)
        fade_out = config.get('fade_out', 0.5)
        
        print(f"\n[{i}/{len(noise_configs)}] 🔧 {noise_name}")
        print(f"     Timing : {start_time:.1f}s → {start_time + duration:.1f}s (durée: {duration:.1f}s)")
        print(f"     SNR : {snr_db} dB (voix {snr_db}dB plus forte que le bruit)")
        print(f"     Offset dans le fichier : {offset:.1f}s")
        print(f"     Fondus : IN {fade_in:.1f}s (log) | OUT {fade_out:.1f}s (exp)")
        
        # Ajouter le bruit avec fondus
        mixed = add_noise_with_fade(
            voice_signal=mixed,
            noise_signal=noise_signals[noise_name],
            sr=sr,
            start_time=start_time,
            duration=duration,
            snr_db=snr_db,
            noise_offset=offset,
            fade_in=fade_in,
            fade_out=fade_out
        )
        
        print(f"     ✓ Bruit ajouté avec succès")
    
    # ÉTAPE 4 : NORMALISATION FINALE
    print(f"\n" + "─"*70)
    max_amplitude = np.max(np.abs(mixed))
    
    if max_amplitude > 0.99:
        print(f"⚠️ Saturation détectée ({max_amplitude:.3f}), normalisation appliquée")
        mixed = mixed / max_amplitude * 0.95
        print(f"   ✓ Signal normalisé à 0.95 ({20*np.log10(0.95):.2f} dB)")
    else:
        print(f"✓ Pas de saturation (peak: {max_amplitude:.3f})")
    
    # ÉTAPE 5 : SAUVEGARDER
    print(f"\n💾 Sauvegarde du fichier final...")
    output_file.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(output_file), mixed, sr)
    
    print(f"   ✓ Fichier sauvegardé : {output_file}")
    print(f"   Durée finale : {len(mixed)/sr:.2f}s")
    print(f"   Format : {sr}Hz, mono")
    
    print("\n" + "="*70)
    print("🎉 MIXAGE TERMINÉ AVEC SUCCÈS !")
    print("="*70)
    
    return mixed, sr


# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION DES 6 BRUITS INDUSTRIELS
# ═══════════════════════════════════════════════════════════════════

# Définir les chemins des fichiers de bruits
NOISE_DIR = DATA_DIR / "audio" / "background_sounds"  # Adapter selon votre structure

# Configuration de chaque bruit industriel
# Format: chaque dictionnaire représente un bruit à ajouter
noise_configurations = [
    {
        'file': NOISE_DIR / 'chalumeau' / 'AV-1-S-OUT-101-1-A.wav',
        'name': 'Chalumeau',
        'start_time': 5.0,      # Commence à 5 secondes de la voix
        'duration': 4.0,        # Dure 4 secondes
        'snr_db': 18,           # Voix 18dB plus forte (bruit discret)
        'offset': 2.0,          # Commence à lire à 2s du fichier bruit
        'fade_in': 0.8,         # Fade in de 0.8s (logarithmique)
        'fade_out': 0.8         # Fade out de 0.8s (exponentielle)
    },
    {
        'file': NOISE_DIR / 'meule' / 'AV-1-S-OUT-201-1-A.wav',
        'name': 'Meule',
        'start_time': 6.0,     # Commence à 6 secondes
        'duration': 3.0,        # Dure 3 secondes
        'snr_db': 15,           # Voix 15dB plus forte (bruit perceptible)
        'offset': 3.0,          # Commence à 3s du fichier
        'fade_in': 1.0,         # Fade in de 1s
        'fade_out': 1.0         # Fade out de 1s
    },
    {
        'file': NOISE_DIR / 'pont_roulant' / 'AV-1-S-OUT-401.wav', 
        'name': 'Pont Roulant',
        'start_time': 2.0,     # Commence à 2 secondes
        'duration': 25.0,        # Dure 25 secondes
        'snr_db': 20,           # Voix 20dB plus forte (bruit de fond)
        'offset': 5.0,          # Commence à 5s (évite le silence)
        'fade_in': 1.2,         # Fade in de 1.2s
        'fade_out': 1.2         # Fade out de 1.2s
    },
    {
        'file': NOISE_DIR / 'rivetage' / 'AV-1-S-MET-403-1.wav', 
        'name': 'Rivetage',
        'start_time': 23.0,     # Commence à 23 secondes
        'duration': 5.5,        # Dure 5.5 secondes
        'snr_db': 12,           # Voix 12dB plus forte (bruit important)
        'offset': 1.0,          # Commence à 1s du fichier
        'fade_in': 0.5,         # Fade in rapide de 0.5s
        'fade_out': 0.7         # Fade out de 0.7s
    },
    {
        'file': NOISE_DIR / 'soudeur' / 'AV-1-S-MET-502.wav', 
        'name': 'Soudeur',
        'start_time': 9.0,     # Commence à 9 secondes
        'duration': 4.0,        # Dure 4 secondes
        'snr_db': 16,           # Voix 16dB plus forte
        'offset': 2.5,          # Commence à 2.5s du fichier
        'fade_in': 0.9,         # Fade in de 0.9s
        'fade_out': 1.1         # Fade out de 1.1s
    },
    {
        'file': NOISE_DIR / 'titanier' / 'AV-1-S-MET-202-1.wav',  
        'name': 'Titanier',
        'start_time': 13.0,     # Commence à 13 secondes
        'duration': 7.0,        # Dure 7 secondes
        'snr_db': 17,           # Voix 17dB plus forte
        'offset': 3.5,          # Commence à 3.5s du fichier
        'fade_in': 1.0,         # Fade in de 1s
        'fade_out': 1.0         # Fade out de 1s
    }
]

# ═══════════════════════════════════════════════════════════════════
# LANCER LE MIXAGE
# ═══════════════════════════════════════════════════════════════════

# Fichiers d'entrée/sortie
voice_file = AUDIO_VOICE_DIR / "ElevenLabs_Spuds_Oxley.mp3"
output_file = OUTPUT_DIR / "mixage_6_bruits_industriels_avec_fondus.wav"

# Lancer le mixage multi-bruits
mixed_signal, sample_rate = mix_multiple_industrial_noises(
    voice_file=voice_file,
    noise_configs=noise_configurations,
    output_file=output_file
)

print(f"\n🎧 Écoutez le résultat : {output_file}")
print(f"📊 Durée totale : {len(mixed_signal)/sample_rate:.2f}s")
print(f"🎯 Règle d'or appliquée sur tous les bruits ✨")


In [ ]:
# Cellule : Diagnostic simple des fichiers manquants
"""
Vérification rapide des fichiers Chalumeau et Rivetage
"""

from pathlib import Path

print("="*70)
print("🔍 DIAGNOSTIC DES FICHIERS MANQUANTS")
print("="*70)

# Les 2 fichiers qui posent problème
fichiers_a_verifier = [
    {
        'nom': 'Chalumeau',
        'chemin': NOISE_DIR / 'chalumeau' / 'AV-1-S-OUT-101-1-A .wav'  # Votre chemin actuel
    },
    {
        'nom': 'Rivetage', 
        'chemin': NOISE_DIR / 'rivetage' / 'AV-1-S-MET-403-1.wav'  # Votre chemin actuel
    }
]

for fichier in fichiers_a_verifier:
    nom = fichier['nom']
    chemin = fichier['chemin']
    
    print(f"\n{'─'*70}")
    print(f"📄 {nom}")
    print(f"{'─'*70}")
    
    # 1. Afficher le chemin qu'on cherche
    print(f"\n🔍 Chemin recherché :")
    print(f"   {chemin}")
    
    # 2. Est-ce que le fichier existe ?
    if chemin.exists():
        print(f"   ✅ Le fichier existe !")
    else:
        print(f"   ❌ Le fichier N'EXISTE PAS")
        
        # 3. Est-ce que le DOSSIER existe ?
        dossier = chemin.parent
        print(f"\n📁 Vérification du dossier parent :")
        print(f"   {dossier}")
        
        if dossier.exists():
            print(f"   ✅ Le dossier existe")
            
            # 4. Lister TOUS les fichiers dans ce dossier
            print(f"\n📂 Contenu du dossier :")
            fichiers_dans_dossier = list(dossier.iterdir())
            
            if len(fichiers_dans_dossier) == 0:
                print(f"   ⚠️  Le dossier est VIDE !")
            else:
                print(f"   {len(fichiers_dans_dossier)} fichier(s) trouvé(s) :\n")
                for f in sorted(fichiers_dans_dossier):
                    icone = "📁" if f.is_dir() else "📄"
                    print(f"   {icone} {f.name}")
                
                # 5. Suggérer les fichiers .wav
                fichiers_wav = [f for f in fichiers_dans_dossier if f.suffix.lower() in ['.wav', '.mp3']]
                if fichiers_wav:
                    print(f"\n💡 Fichiers audio trouvés (utilisez un de ces noms) :")
                    for f in fichiers_wav:
                        print(f"   → {f.name}")
        else:
            print(f"   ❌ Le dossier N'EXISTE PAS")
            print(f"\n💡 Vérifiez que le dossier '{dossier.name}' existe dans :")
            print(f"   {dossier.parent}")

print(f"\n{'='*70}")
print("FIN DU DIAGNOSTIC")
print("="*70)
